# Dataplex: Metadata Feed API Demo (February 2026 Preview)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maruti123/POH/blob/main/dataplex_change_feed_demo.ipynb)

This notebook demonstrates how to programmatically create and configure a **Metadata Feed** using the Dataplex API. This feature enables real-time tracking of metadata changes (Create, Update, Delete) via Pub/Sub.

## Use Case
A Data Governance team needs to trigger automated security scans whenever a new BigQuery table is created or an existing one is modified. We'll use the API to set up this event-driven workflow.

### Requirements
- Dataplex API enabled.
- A Pub/Sub Topic to receive notifications.
- Dataplex Service Account with `roles/pubsub.publisher` permissions.

In [ ]:
from google.colab import auth
auth.authenticate_user()

import os
project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
location = 'us-central1' 
topic_id = 'metadata-notifications-topic'
feed_id = 'central-governance-feed'

### 2. Configure Metadata Feed JSON

The `metadataFeeds` resource requires a `pubsub_topic` and a `scope` (the project, folder, or organization being monitored).

In [ ]:
# Example of the 'MetadataFeed' resource structure
feed_config = {
    "name": f"projects/{project_id}/locations/{location}/metadataFeeds/{feed_id}",
    "pubsub_topic": f"projects/{project_id}/topics/{topic_id}",
    "scope": {
        "project": f"projects/{project_id}"
    },
    "condition": "resource.type == 'bigquery-table'" # Optional filter for specific resources
}

print("Preparing Dataplex Metadata Feed Configuration...")
print(f"Monitoring Scope: {feed_config['scope']['project']}")
print(f"Target Pub/Sub: {feed_config['pubsub_topic']}")
print("Success: Metadata feed prepared for API deployment.")

### 3. Handling Notifications (Pub/Sub Callback)

The feed sends a structured message for every metadata event. You can then route these to Cloud Functions for automated processing.

In [ ]:
import json

def process_notification(data):
    event = json.loads(data)
    print(f"--- METADATA EVENT DETECTED ---")
    print(f"Resource: {event['resource']}")
    print(f"Action: {event['change_type']}") # e.g., UPDATE, CREATE
    print(f"Triggering Automated Security Scan...")

mock_event = json.dumps({
    "resource": "projects/my-prod/datasets/finance/tables/salaries",
    "change_type": "UPDATE",
    "timestamp": "2026-03-09T10:00:00Z"
})

process_notification(mock_event)

### 4. Things to remember or know
- **Publisher Permissions**: You MUST grant `service-PROJECT_NUMBER@gcp-sa-dataplex.iam.gserviceaccount.com` the `Pub/Sub Publisher` role on your topic.
- **Near Real-Time**: Notifications are typically delivered within seconds of the metadata update.
- **Broad Scope**: A single feed can monitor an entire Organization or Folder, simplifying large-scale governance.